# SmartCommerce AI — Exploratory Data Analysis & Cleaning
This notebook documents the data loading, exploration, cleaning decisions, and feature engineering steps taken on the Olist Brazilian E-Commerce dataset to prepare it for sales forecasting (Prophet) and customer segmentation (K-Means).

## 1. Environment Setup & Data Loading
We start by loading pandas, numpy, and checking the raw Olist dataset files downloaded into `data/raw/`.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW_DIR = Path("../raw")
PROCESSED_DIR = Path("../processed")

print("Loading datasets...")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

print(f"Customers: {customers.shape}")
print(f"Products: {products.shape}")
print(f"Orders: {orders.shape}")
print(f"Order Items: {order_items.shape}")
print(f"Payments: {payments.shape}")

ModuleNotFoundError: No module named 'seaborn'

## 2. Exploratory Inspection
Let's print standard info and null counts of the datasets.

In [ ]:
print("--- Missing values in orders ---")
print(orders.isnull().sum())

print("\n--- Missing values in products ---")
print(products.isnull().sum())

## 3. Data Cleaning Decisions

### 3.1. Customers Table
We only keep `customer_id`, `customer_city`, and `customer_state` to match the PostgreSQL schema.

In [ ]:
cleaned_customers = customers[["customer_id", "customer_city", "customer_state"]].copy()
cleaned_customers.head(3)

### 3.2. Products Table (Category Translation)
We translate Portuguese category names to English and map missing categories to `"unknown"`.

In [ ]:
products_translated = products.merge(translation, on="product_category_name", how="left")
products_translated["product_category_name_english"] = products_translated["product_category_name_english"].fillna("unknown")
cleaned_products = pd.DataFrame({
    "product_id": products_translated["product_id"],
    "product_category": products_translated["product_category_name_english"]
})
print("Products null categories count:", cleaned_products["product_category"].isnull().sum())
cleaned_products.head(3)

### 3.3. Deduplicating Order Items (Primary Key Constraints)
PostgreSQL uses `PRIMARY KEY (order_id, product_id)` in `order_items`. To prevent key violations for duplicate item purchases, we group by order and product and sum the price and freight.

In [ ]:
print(f"Raw order items count: {len(order_items)}")
cleaned_order_items = order_items.groupby(["order_id", "product_id"], as_index=False).agg({
    "price": "sum",
    "freight_value": "sum"
})
print(f"Deduplicated order items count: {len(cleaned_order_items)}")

## 4. Forecasting Feature Generation (Daily Sales)
We build the daily sales time series for Prophet. We filter out canceled/unavailable orders and restrict dates to the stable timeline (Jan 2017 to Aug 2018).

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])].copy()
sales_merged = valid_orders.merge(order_items, on="order_id", how="inner")
sales_stable = sales_merged[
    (sales_merged["order_purchase_timestamp"] >= "2017-01-01") & 
    (sales_merged["order_purchase_timestamp"] <= "2018-08-31")
]
sales_stable["date"] = sales_stable["order_purchase_timestamp"].dt.date
daily_sales = sales_stable.groupby("date")["price"].sum().reset_index()
daily_sales.columns = ["ds", "y"]

# Plot Daily Sales Evolution
plt.figure(figsize=(12, 5))
plt.plot(daily_sales["ds"], daily_sales["y"], color="#3f51b5")
plt.title("Daily Sales Revenue (Olist 2017-2018)")
plt.xlabel("Date")
plt.ylabel("Revenue (€)")
plt.grid(True)
plt.show()

## 5. Customer Segmentation Feature Generation (RFM)
For customer clustering, we aggregate metrics per physical client (`customer_unique_id`).

In [ ]:
orders_valid = orders[~orders["order_status"].isin(["canceled", "unavailable"])].copy()
order_values = order_items.groupby("order_id")["price"].sum().reset_index()
orders_with_value = orders_valid.merge(order_values, on="order_id", how="inner")
customers_mapped = orders_with_value.merge(customers, on="customer_id", how="inner")

baseline_date = orders_valid["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
rfm = customers_mapped.groupby("customer_unique_id").agg({
    "order_purchase_timestamp": lambda x: (baseline_date - x.max()).days,
    "order_id": "nunique",
    "price": "sum"
}).reset_index()
rfm.columns = ["customer_unique_id", "recency", "frequency", "monetary"]
rfm.describe()

### 5.1. Distributions of RFM Metrics

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rfm["recency"], bins=30, kde=True, ax=ax[0], color="orange")
ax[0].set_title("Recency Distribution")
sns.histplot(rfm["frequency"], bins=10, ax=ax[1], color="green")
ax[1].set_title("Frequency Distribution")
sns.histplot(rfm[rfm["monetary"] < 500]["monetary"], bins=30, kde=True, ax=ax[2], color="purple")
ax[2].set_title("Monetary (< 500) Distribution")
plt.tight_layout()
plt.show()

## 6. Conclusion
All datasets have been successfully cleaned, transformed, and saved. The features are fully aligned with the requirements for offline modeling and database schema constraints.